In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import scipy.sparse as sp
import yaml


import anndata as an
import scanpy as sc
import rapids_singlecell as rsc
import scvi

/nfs/turbo/umms-indikar/Cooper/conda_envs/scrapids/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/nfs/turbo/umms-indikar/Cooper/conda_envs/scrapids/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_categorical_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/nfs/turbo/umms-indikar/Cooper/conda_envs/scrapids/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_continuous_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


In [2]:
dpath = "/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/"

file_list = glob.glob(f"{dpath}hsc_a*/*.h5ad")
file_list

['/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/hsc_aux/lymphoid_cells.h5ad',
 '/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/hsc_anndata/hsc_and_progenitors.h5ad',
 '/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/hsc_anndata/fibroblasts.h5ad',
 '/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/hsc_anndata/mesenchymal_cells.h5ad',
 '/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/hsc_anndata/myeloid_cells.h5ad',
 '/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/hsc_anndata/innate_lymphoid_cells.h5ad',
 '/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/hsc_anndata/endothelial_cells.h5ad']

In [3]:
def load_first_n_cells(file: str, n: int) -> an.AnnData:
    with h5py.File(file, 'r') as f:
        if isinstance(f['X'], h5py.Group):  # likely sparse
            data = f['X/data'][:]
            indices = f['X/indices'][:]
            indptr = f['X/indptr'][:n+1]  # one extra for CSR row pointer
            shape = (n, f['X'].attrs['shape'][1])

            data = data[:indptr[-1]]  # only use data for first n rows
            indices = indices[:indptr[-1]]

            X_slice = sp.csr_matrix((data, indices, indptr), shape=shape)
        else:
            X_slice = f['X'][:n]

    backed = an.read_h5ad(file, backed='r')
    obs = backed.obs.iloc[:n].copy()
    var = backed.var.copy()
    backed.file.close()

    return an.AnnData(X=X_slice, obs=obs, var=var)

In [ ]:
sample_size = 100000
adata_list = []

for fpath in file_list:
    basename = os.path.basename(fpath).replace(".h5ad", "")
    print(f"Working {basename}...")
    adata = sc.read_h5ad(fpath)
    # adata = read_top_n_anndata(fpath, sample_size)
    print(f"\t {len(adata)} Cells")
    adata.obs['basename'] = basename
    adata.var_names = adata.var['feature_name']

    if not sample_size is None:
        if sample_size >= len(adata):
            print(f"\tTaking all cells")
        else:
            print(f"\tSampled {sample_size} ({(sample_size / len(adata)) * 100:.2f}%)")
            adata = sc.pp.sample(adata, n=sample_size, copy=True)
    
    adata_list.append(adata)
    # break

adata = an.concat(adata_list)
adata.obs['dataset_id_int'] = adata.obs['dataset_id'].astype('category').cat.codes
rsc.get.anndata_to_GPU(adata) # move to GPU
adata.obs['basename'].value_counts()
print()
adata

Working lymphoid_cells...


# Basic filtering and processing

In [ ]:
rsc.pp.filter_cells(adata, min_counts=200)
rsc.pp.filter_genes(adata, min_counts=50)

adata.layers['raw'] = adata.X.get()
rsc.get.anndata_to_GPU(adata) # move to GPU

rsc.pp.normalize_total(adata, target_sum=1e4)
rsc.pp.log1p(adata)
adata


# Cell type simplification

In [ ]:
# Load YAML into a dictionary
with open("../resources/cell_map.yaml", "r") as file:
    cell_map = yaml.safe_load(file)

adata.obs['cell_type_simplified'] = adata.obs['cell_type'].map(cell_map)
adata.obs['cell_type_simplified'] = adata.obs['cell_type_simplified'].astype('category')
adata.obs['cell_type_simplified'].value_counts(dropna=False)

In [ ]:
# drop nan
adata = adata[adata.obs['cell_type_simplified'].notna(), :].copy()
adata

# Feature Engineering

In [ ]:
adata.raw = adata  # keep full dimension safe
print(f"Number of genes before HVG selection: {adata.n_vars}")

rsc.pp.highly_variable_genes(
    adata, 
    n_top_genes=5000, 
    flavor="seurat_v3", 
    batch_key="basename",
)

adata = adata[:, adata.var['highly_variable']].copy()

print(f"Number of genes after HVG selection: {adata.n_vars}")

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 5, 4
sc.pl.highly_variable_genes(
    adata
)

adata

In [ ]:
rsc.pp.pca(
    adata,
)

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 5, 3
sc.pl.pca_variance_ratio(
    adata
)


# Embedding

In [ ]:
rsc.pp.neighbors(
    adata,
)

rsc.tl.umap(
    adata,
)

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 6, 6

sc.pl.umap(
    adata,
    ncols=1,
    color=['basename', 'cell_type_simplified', 'dataset_id_int']
)


# Clustering

In [ ]:
rsc.tl.leiden(
    adata,
    resolution=0.35,
)

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 6, 6

sc.pl.umap(
    adata,
    ncols=1,
    color=['leiden', 'cell_type_simplified']
)

# scVI

In [ ]:
adata

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata, 
    layer="raw",
    batch_key="dataset_id",
    labels_key='cell_type_simplified',
)

model = scvi.model.SCVI(
    adata,
    n_layers=2,
    n_latent=24,
    gene_likelihood="nb",
)

model.train(
    max_epochs=100,
    accelerator="gpu",
    devices="auto",
    enable_model_summary=True,
    batch_size=1000,
    early_stopping=True,
    early_stopping_patience=2,
    early_stopping_monitor='validation_loss',
)

adata.obsm["X_scVI"] = model.get_latent_representation()

In [ ]:
rsc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=55)
rsc.tl.leiden(adata)

rsc.tl.umap(adata)

sc.pl.umap(
    adata,
    color=["leiden", "basename", "cell_type_simplified", "dataset_id_int"],
    frameon=False,
    ncols=1,
)

# scANVI

In [ ]:
scanvi_model = scvi.model.SCANVI.from_scvi_model(
    model,
    adata=adata,
    labels_key="cell_type_simplified",
    unlabeled_category="Unknown",
)

scanvi_model.train(
    max_epochs=20, 
    n_samples_per_label=100,
    accelerator="gpu",
    devices="auto",
    enable_model_summary=True,
    batch_size=1000,
    early_stopping=True,
    early_stopping_patience=2,
    early_stopping_monitor='validation_loss',
)

adata.obsm['X_scANVI'] = scanvi_model.get_latent_representation(adata)

In [ ]:
rsc.pp.neighbors(adata, use_rep='X_scANVI', n_neighbors=55)
rsc.tl.leiden(adata)

rsc.tl.umap(adata)

sc.pl.umap(
    adata,
    color=["leiden", "basename", "cell_type_simplified", "dataset_id_int"],
    frameon=False,
    ncols=1,
)

In [ ]:
break

# differential gene expression

In [ ]:
rsc.tl.rank_genes_groups_logreg(
    adata,
    groupby='basename',
    max_iter=100,
)


df = sc.get.rank_genes_groups_df(
    adata,
    group=None,
)

df.head()


In [ ]:
rsc.__version__